In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import torch
import gc
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report
from preprocessing import get_preprocessing_attributes

In [ ]:
data = pd.read_csv("./dataset/data.csv")
data.drop_duplicates(subset=['title'], inplace=True)

In [ ]:
data_train, data_val = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
data_train_preprocessed = get_preprocessing_attributes(data_train)
data_val_preprocessed = get_preprocessing_attributes(data_val)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
svd = TruncatedSVD(n_components=128, random_state=42)

X_train_tfidf = tfidf_vectorizer.fit_transform(data_train_preprocessed['preprocessed_text'])
X_train_tfidf_features = svd.fit_transform(X_train_tfidf)

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
deberta_model = AutoModel.from_pretrained("microsoft/deberta-v3-small")

def get_embeddings(data, batch_size=4):
    texts = data.tolist() if hasattr(data, 'tolist') else list(data)
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts, 
            padding=True, 
            truncation=True, 
            max_length=128, 
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = deberta_model(**inputs)
            
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
        del inputs, outputs
        gc.collect()

    return np.vstack(all_embeddings)

X_deberta_features = get_embeddings(data_train_preprocessed['preprocessed_text'])
X_train_mixed_data = np.hstack((X_deberta_features, X_train_tfidf_features))

model = Sequential([
    Input(shape=(X_train_mixed_data.shape[1],)),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(2, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train_mixed_data, data_train['label'], epochs=10, batch_size=2)

In [ ]:
# print(classification_report(data_val_preprocessed['label'], linear_SVC_TFIDF_pred, target_names=['real', 'fake'], digits=4))